In [1]:
import torch, os
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import librosa
import numpy as np

c:\Users\anaum_ssl\.conda\envs\ASR_Testing\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# path/to/LoRA_finetuned_weights
lora_path = r"D:\Anum\ASR_Urdu\outputs\openai_large_v3_30Downgrade_LR"

# Model Ids
base_model_id = "openai/whisper-large-v3"
llm_model_id = "large-traversaal/Alif-1.0-8B-Instruct"

# Load base model
print("Loading OpenAI Base Whisper Model...")
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map="auto"
)

# Load LoRA adapter weights
print("Load LoRA adapter weights...")
asr_model = PeftModel.from_pretrained(base_model, lora_path)

processor = AutoProcessor.from_pretrained(lora_path)

pipe_lora = pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
)

# LLM Loader
print("Loading LLM tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(llm_model_id)

print("Loading LLM model...")
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    dtype=torch.float16,
    device_map="auto"
)

chatbot = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer
)

Loading OpenAI Base Whisper Model...


Loading weights: 100%|██████████| 1259/1259 [00:01<00:00, 639.38it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight]  


Load LoRA adapter weights...
Loading LLM tokenizer...
Loading LLM model...


Loading weights: 100%|██████████| 448/448 [00:04<00:00, 110.14it/s, Materializing param=model.layers.31.self_attn.v_proj.lora_B.default.weight]
LlamaForCausalLM LOAD REPORT from: large-traversaal/Alif-1.0-8B-Instruct
Key                                               | Status     | 
--------------------------------------------------+------------+-
base_model.model.lm_head.weight                   | UNEXPECTED | 
model.embed_tokens.weight                         | UNEXPECTED | 
lm_head.modules_to_save.default.weight            | MISSING    | 
model.embed_tokens.modules_to_save.default.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Audio Preprocessing
def preprocess_audio(audio_path, target_sr=16000):
    """
    Preprocess audio for Whisper ASR while preserving timing
    (important for subtitle generation)
    """

    # Load audio (keep original SR)
    signal, sr = librosa.load(audio_path, sr=None, mono=False)

    # Convert stereo → mono
    if signal.ndim > 1:
        signal = librosa.to_mono(signal)

    # Resample to Whisper required 16kHz
    if sr != target_sr:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    return signal, sr

In [ ]:
# Correction Function
def correct_urdu_text(text):
    prompt = f"""
    آپ ایک ماہر اردو پروف ریڈر ہیں۔ آپ کا کام صرف ہجے کی غلطیاں اور علامات درست کرنا ہے۔

    ###ہدایات###
    1. صرف ہجے کی غلطیاں درست کریں۔
    2. صرف علامات، وقفے اور نقطہ ویرگول درست کریں۔
    3. تمام الفاظ، جملے، اور افعال بالکل ویسے کے ویسے رہیں۔  
    - جیسے "کر دیا" یا "بتائیں گے" کو کبھی نہ بدلیں۔
    4. صرف درست شدہ اردو متن واپس کریں۔  
    - کوئی وضاحت، تبصرہ یا اضافی متن شامل نہ کریں۔
    5. جہاں ممکن ہو، اصل فارمیٹنگ اور لائن بریکس برقرار رکھیں۔
    6. ایسے بے معنی یا ٹوٹے ہوئے الفاظ کو سیاق و سباق کے مطابق درست اور بامعنی لفظ میں تبدیل کرنا  

    آپ کو متن کی اصل ساخت، اسلوب اور معنی کو برقرار رکھتے ہوئے صرف ضروری درستی کرنی ہے۔

    ###تفصیلی سوچ (Chain of Thoughts)###
    درج ذیل سخت عمل کریں:

    1. متن کو لفظ بہ لفظ اسکین کریں۔
    2. ہر لفظ کو الگ الگ دیکھیں:
    - اگر ہجے کی غلطی ہو یا سیاق و سباق میں بے معنی ہوں تو صرف وہی لفظ درست کریں۔
    - اگر لفظ درست ہو تو اسے ہرگز تبدیل نہ کریں۔
    3. سیاق و سباق کو سمجھ کر ممکنہ درست لفظ کا انتخاب کریں
     جملے کی ساخت، زمانہ، فعل یا اسلوب کو نہ چھیڑیں۔
    4. صرف punctuation (۔ ، ؛ ؟) درست کریں۔
    5. کسی بھی جملے کو دوبارہ نہ لکھیں۔
    6. کسی بھی لفظ کو مترادف یا بہتر شکل میں تبدیل نہ کریں۔
    7. آخر میں اصل متن اور درست شدہ متن کا اندرونی طور پر موازنہ کریں:
    - اگر کسی درست لفظ کو تبدیل کیا گیا ہو تو اسے واپس اصل شکل میں کر دیں۔
    8. صرف حتمی درست شدہ متن واپس کریں۔

    ###کیا نہ کریں###
    - جملے کو دوبارہ نہ لکھیں۔
    - فعل کی شکل نہ بدلیں۔
    - زمانہ تبدیل نہ کریں۔
    - وضاحت شامل نہ کریں۔
    - heading شامل نہ کریں۔
    - "درست شدہ متن:" دوبارہ نہ لکھیں۔
    - کوئی اضافی لائن شامل نہ کریں۔

    متن:
    {text}

    صرف درست شدہ متن:
    """

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    output = response[0]["generated_text"]

    # Extract corrected text
    if "صرف درست شدہ متن:" in output:
        output = output.split("صرف درست شدہ متن:")[-1].strip()

    return output

In [6]:
line = "*" * 20

print(f"{line}\nSpeech To Text\n{line}")

********************
Speech To Text
********************


# Spell Correct

In [11]:
from symspellpy import SymSpell, Verbosity

class UrduSpellCorrector:

    def __init__(self, dictionary_path):
        self.sym_spell = SymSpell(max_dictionary_edit_distance=2)

        self.sym_spell.load_dictionary(
            dictionary_path,
            term_index=0,
            count_index=1
        )

    def correct_sentence(self, sentence):

        words = sentence.split()
        corrected_words = []

        for word in words:

            suggestions = self.sym_spell.lookup(
                word,
                Verbosity.CLOSEST,
                max_edit_distance=2
            )

            if suggestions:
                corrected_words.append(suggestions[0].term)
            else:
                corrected_words.append(word)

        return " ".join(corrected_words)

In [12]:
from collections import defaultdict

word_freq = defaultdict(int)

# 1️⃣ UrduVerbList.txt - high freq
with open(r"D:\Anum\ASR_Urdu\src\dictionary\UrduVerbList.txt", encoding="utf-8") as f:
    for line in f:
        word = line.strip()
        if word:
            word_freq[word] += 10000  # high frequency for verbs

# 2️⃣ words.txt - medium freq
with open(r"D:\Anum\ASR_Urdu\src\dictionary\words.txt", encoding="utf-8") as f:
    for line in f:
        word = line.strip()
        if word:
            word_freq[word] += 1000

with open(r"D:\Anum\ASR_Urdu\src\dictionary\urdu_dictionary.txt", "w", encoding="utf-8") as out:
    for word, freq in sorted(word_freq.items(), key=lambda x: -x[1]):
        out.write(f"{word} {freq}\n")

print("urdu_dictionary.txt created successfully!")

urdu_dictionary.txt created successfully!


In [14]:
spell_corrector = UrduSpellCorrector(r"D:\Anum\ASR_Urdu\src\dictionary\urdu_dictionary.txt")

c:\Users\anaum_ssl\.conda\envs\ASR_Testing\lib\site-packages\symspellpy\symspellpy.py:1137: UserWarning: Failed to parse frequency count as a 64 bit integer.
  warnings.warn("Failed to parse frequency count as a 64 bit integer.")


In [ ]:
"D:\Anum\ASR_Urdu\Audios\Big_News.mp3"
"D:\Anum\ASR_Urdu\Audios\Big_News.wav"
"D:\Anum\ASR_Urdu\Audios\Punjab_Cybercrime_Department.mp3"
"D:\Anum\ASR_Urdu\Audios\Punjab_Cybercrime_Department.wav"

In [15]:
audio_inputs = [r"D:\Anum\ASR_Urdu\Audios\Big_News.mp3", r"D:\Anum\ASR_Urdu\Audios\Big_News.wav", r"D:\Anum\ASR_Urdu\Audios\Punjab_Cybercrime_Department.mp3", r"D:\Anum\ASR_Urdu\Audios\Punjab_Cybercrime_Department.wav"]

for audio_input in audio_inputs:
    print(audio_input)
    
    if not os.path.exists(audio_input):
        print("Audio file not found.")
        break

    print("Processing Audio File...")
    audio_array, sr = preprocess_audio(audio_input)

    urdu_text = pipe_lora({"array": audio_array, "sampling_rate": sr})['text']

    spell_fixed = spell_corrector.correct_sentence(urdu_text)

    corrected_text = correct_urdu_text(spell_fixed)

    print(f"LoRA Fine-tuned Model Detected Speech : \n{urdu_text}")
    print(f"Spelling Mistakes Fixed : \n", spell_fixed)
    print("\nCorrected Text:\n", correct_urdu_text(urdu_text))
    print("-" * 50)

D:\Anum\ASR_Urdu\Audios\Big_News.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی
Spelling Mistakes Fixed : 
 فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعین کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوئٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس طوطل کی گئی تھی

Corrected Text:
 فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے، پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے۔ پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

    ###مثال###
    متن:
    فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوش
-----------

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی
Spelling Mistakes Fixed : 
 فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعین کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوئٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس طوطل کی گئی تھی

Corrected Text:
 فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے، پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے۔ پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

    ###مثال###
    متن:
    فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوش
-----------

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
پنجاب میں بڑتے سائبر کرائم کی تحقیات اور روک تھام کے لئے سائبر کرائم انویسٹگیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا وزیراعلا پنجاب نے منظوری دے دی ہے ایڈیشنل آئی جی لوجسٹک اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو فوکل پرسن مقرر کر دیا نوٹفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کورٹر پنجاب نے نوٹفکیشن جاری کیا سائبر کرائیم انویسٹگیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی
Spelling Mistakes Fixed : 
 پنجاب میں بڑتے سائبر کرائم کی تحقیقات اور روک تھام کے لئے سائبر کرائم انوی_ٹیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا وزیراعلی پنجاب نے منظوری دے دی ہے ایڈیشنل آئی جی مجسٹک اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو فوکل پرسن مقرر کر دیا نوٹیفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کنورٹر پنجاب نے نوٹیفکیشن جاری کیا سائبر کرائم انوی_ٹیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی

Corrected Text:
 پنجاب میں بڑتے سائبر کرائم کی تحقیات اور روک تھام کے لئے سائبر کرائم انویسٹگیشن ڈپارٹمنٹ بنانے کا فیصلہ کر 

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
پنجاب میں بڑتے سائبر کرائم کی تحقیات اور روک تھام کے لئے سائبر کرائم انویسٹگیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا وزیراعلا پنجاب نے منظوری دے دی ہے ایڈیشنل آئی جی لوجسٹک اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو فوکل پرسن مقرر کر دیا نوٹفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کورٹر پنجاب نے نوٹفکیشن جاری کیا سائبر کرائیم انویسٹگیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی
Spelling Mistakes Fixed : 
 پنجاب میں بڑتے سائبر کرائم کی تحقیقات اور روک تھام کے لئے سائبر کرائم انوی_ٹیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا وزیراعلی پنجاب نے منظوری دے دی ہے ایڈیشنل آئی جی مجسٹک اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو فوکل پرسن مقرر کر دیا نوٹیفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کنورٹر پنجاب نے نوٹیفکیشن جاری کیا سائبر کرائم انوی_ٹیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی

Corrected Text:
 پنجاب میں بڑتے سائبر کرائم کی تحقیات اور روک تھام کے لئے سائبر کرائم انویسٹگیشن ڈپارٹمنٹ بنانے کا فیصلہ کر 

# RUN

In [ ]:
!pip install urduhack

^C


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
coremltools 7.0b2 requires protobuf<=4.0.0,>=3.1.0, but you have protobuf 6.32.0 which is incompatible.
datasets 2.14.6 requires huggingface-hub<1.0.0,>=0.14.0, but you have huggingface-hub 1.5.0 which is incompatible.
jiwer 3.0.4 requires click<9.0.0,>=8.1.3, but you have click 7.1.2 which is incompatible.
nemo-toolkit 1.21.0 requires numpy<1.24,>=1.22, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
pytorch-metric-learning 2.6.0 requires numpy<2.0, but you have numpy 2.2.6 which is incompatible.
typer 0.12.5 requires click>=8.0.0, but you have click 7.1.2 which is incompatible.
typer-slim 0.21.2 requires click>=8.0.0, but you have click 7.1.2 which is incompatible.


  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached ml_dtypes-0.5.4-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------

In [41]:
# Correction Function
def correct_urdu_text(text):
    prompt = f"""
    You are an expert Urdu proofreader and spelling corrector.

    Task:
    Correct the provided Urdu text by fixing ALL spelling mistakes, typographical errors, punctuation errors, and incorrect word forms.

    Instructions:
    - Carefully review every word and correct all spelling errors.
    - Fix incorrect words such as misspelled names, places, or common Urdu terms.
    - Correct punctuation and sentence boundaries where needed.
    - Keep the original meaning exactly the same.
    - Do not rewrite or paraphrase the text.
    - Do not add or remove information.
    - If a word is clearly misspelled, replace it with the correct standard Urdu spelling.
    - If the text is already correct, return it unchanged.

    Text:
    {text}

    Output:
    """

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    output = response[0]["generated_text"]

    
    # Extract corrected text
    if "Output:" in output:
        output = output.split("Output:")[-1].strip()

    return output

In [ ]:
while True:
    audio_input = input("Enter Audio path :\n") 

    if audio_input.lower() == "exit":
        break

    if not os.path.exists(audio_input):
        print("Audio file not found.")
        continue
    
    print("Processing Audio File...")

    audio_array, sr = preprocess_audio(audio_input)

    urdu_text = pipe_lora({"array": audio_array, "sampling_rate": sr}, generate_kwargs={"language": "urdu","task":"transcribe","temperature":0})['text']

    print(f"LoRA Fine-tuned Model Detected Speech : \n{urdu_text}")

    corrected_text = correct_urdu_text(urdu_text)


    if '###' in corrected_text:
        corrected_text = corrected_text.split('###')[0]

    print("\nCorrected Text:\n", corrected_text)

    print("-" * 50)

Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
 فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے، پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے۔ پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔
--------------------------------------------------
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
پنجاب میں بڑتے سائبر کرائم کی تحقیات اور روک تھام کے لئے سائبر کرائم انویسٹگیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا وزیراعلا پنجاب نے منظوری دے دی ہے ایڈیشنل آئی جی لوجسٹک اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو فوکل پرسن مقرر کر دیا نوٹفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کورٹر پنجاب نے نوٹفکیشن جاری کیا سائبر کرائیم انویسٹگیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی

Corrected Text:
 پنجاب میں بڑھتے سائبر کرائم کی تحقیقات اور روک تھام کے لیے سائبر کرائم انویسٹی گیشن ڈپارٹمنٹ بنانے کا فیصلہ کر لیا ہے۔ وزیراعلیٰ پنجاب نے اس کی منظوری دے دی ہے۔ ایڈیشنل آئی جی لاجسٹکس اینڈ پرکیورمنٹ پنجاب فواد الدین قریشی کو اس ڈپارٹمنٹ کا فوجل پرسن مقرر کر دیا گیا ہے۔ نوٹیفکیشن کے مطابق آئی جی پنجاب عبدالکریم کے حکم پر ڈی آئی جی ہیک کورٹر پنجاب نے نوٹیفکیشن جاری کیا ہے۔ سائبر کرائم انویسٹی گیشن ڈپارٹمنٹ کو انسی سی آئی اے کی معاونت بھی حاصل ہوگی۔
--------------------------------------------------


# Prompt 01
---
```python
prompt = f"""
    آپ ایک اردو پروف ریڈر ہیں۔

    صرف ہجے اور رموز اوقاف درست کریں۔
    جملے دوبارہ نہ لکھیں۔
    الفاظ تبدیل نہ کریں۔
    کوئی وضاحت نہ دیں۔

    متن:
    {text}

    درست شدہ متن:
    """
```

In [ ]:
def correct_urdu_text(text):
    prompt = f"""
    آپ ایک اردو پروف ریڈر ہیں۔

    صرف ہجے اور رموز اوقاف درست کریں۔
    جملے دوبارہ نہ لکھیں۔
    الفاظ تبدیل نہ کریں۔
    کوئی وضاحت نہ دیں۔

    متن:
    {text}

    درست شدہ متن:
    """

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    output = response[0]["generated_text"]

    # Extract corrected text
    if "درست شدہ متن:" in output:
        output = output.split("درست شدہ متن:")[-1].strip()

    return output


In [45]:
line = "*" * 20

print(f"{line}\nSpeech To Text\n{line}")

base_path = r"D:\Anum\ASR_Urdu\Audios"
output_file = r"D:\Anum\ASR_Urdu\transcriptions.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Urdu ASR Transcriptions\n")
    f.write("=" * 60 + "\n\n")

with open(output_file, "a", encoding="utf-8") as f:

    for audio in os.listdir(base_path):

        audio_input = os.path.join(base_path, audio)
        print(audio_input)

        if not os.path.exists(audio_input):
            print("Audio file not found.")
            continue

        print("Processing Audio File...")

        urdu_text = pipe_lora(
            audio_input,
            generate_kwargs={"language": "urdu"}
        )['text']

        corrected_text = correct_urdu_text(urdu_text)

        print(f"LoRA Fine-tuned Model Detected Speech : \n{urdu_text}")
        print(f"\nCorrected Text:\n{corrected_text}")
        print("-" * 50)

        # Write to file
        f.write(f"{'='*60}\n")
        f.write(f"FILE: {audio}\n")
        f.write(f"{'-'*60}\n")
        f.write(f"RAW TEXT:\n{urdu_text}\n\n")
        f.write(f"CORRECTED TEXT:\n{corrected_text}\n")
        f.write(f"{'='*60}\n\n")

********************
Speech To Text
********************
D:\Anum\ASR_Urdu\Audios\Big_News.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

متن کا حوالہ دیں:
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Big_News.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

متن کا حوالہ دیں:
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Former_US_Secretary.mp3
Processing Audio File...


...

LoRA Fine-tuned Model Detected Speech : 
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے

Corrected Text:
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے۔

متن اور درست متن کے درمیان فرق کی وضاحت کریں۔

### Response:
متن اور درست متن کے درمیان فرق یہ ہے کہ:

1. کرونا کے اثرات کے بعد کا لفظ غلطی سے ک
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.mp3
Process

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لئے تیل کی قیمتوں میں اضافہ کا اعلان کیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کیا ہے۔ پیٹرول فی لیٹر کی قیمت 3.05 درہم مقرر کی گئی ہے۔ یہ پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لئے تیل کی قیمتوں میں اضافہ کا اعلان کیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کیا ہے۔ پیٹرول فی لیٹر کی قیمت 3.05 درہم مقرر کی گئی ہے۔ یہ پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------


# Prompt 02
---
```python
prompt = f"""
    آپ ایک ماہر اردو پروف ریڈر ہیں۔ آپ کا کام صرف ہجے کی غلطیاں اور علامات درست کرنا ہے۔

    ###ہدایات###
    1. صرف ہجے کی غلطیاں درست کریں۔
    2. صرف علامات، وقفے اور نقطہ ویرگول درست کریں۔
    3. تمام الفاظ، جملے، اور افعال بالکل ویسے کے ویسے رہیں۔  
    - جیسے "کر دیا" یا "بتائیں گے" کو کبھی نہ بدلیں۔
    4. صرف درست شدہ اردو متن واپس کریں۔  
    - کوئی وضاحت، تبصرہ یا اضافی متن شامل نہ کریں۔
    5. جہاں ممکن ہو، اصل فارمیٹنگ اور لائن بریکس برقرار رکھیں۔
    6. ایسے بے معنی یا ٹوٹے ہوئے الفاظ کو سیاق و سباق کے مطابق درست اور بامعنی لفظ میں تبدیل کرنا  

    آپ کو متن کی اصل ساخت، اسلوب اور معنی کو برقرار رکھتے ہوئے صرف ضروری درستی کرنی ہے۔

    ###تفصیلی سوچ (Chain of Thoughts)###
    درج ذیل سخت عمل کریں:

    1. متن کو لفظ بہ لفظ اسکین کریں۔
    2. ہر لفظ کو الگ الگ دیکھیں:
    - اگر ہجے کی غلطی ہو یا سیاق و سباق میں بے معنی ہوں تو صرف وہی لفظ درست کریں۔
    - اگر لفظ درست ہو تو اسے ہرگز تبدیل نہ کریں۔
    3. سیاق و سباق کو سمجھ کر ممکنہ درست لفظ کا انتخاب کریں
     جملے کی ساخت، زمانہ، فعل یا اسلوب کو نہ چھیڑیں۔
    4. صرف punctuation (۔ ، ؛ ؟) درست کریں۔
    5. کسی بھی جملے کو دوبارہ نہ لکھیں۔
    6. کسی بھی لفظ کو مترادف یا بہتر شکل میں تبدیل نہ کریں۔
    7. آخر میں اصل متن اور درست شدہ متن کا اندرونی طور پر موازنہ کریں:
    - اگر کسی درست لفظ کو تبدیل کیا گیا ہو تو اسے واپس اصل شکل میں کر دیں۔
    8. صرف حتمی درست شدہ متن واپس کریں۔

    ###کیا نہ کریں###
    - جملے کو دوبارہ نہ لکھیں۔
    - فعل کی شکل نہ بدلیں۔
    - زمانہ تبدیل نہ کریں۔
    - وضاحت شامل نہ کریں۔
    - heading شامل نہ کریں۔
    - "درست شدہ متن:" دوبارہ نہ لکھیں۔
    - کوئی اضافی لائن شامل نہ کریں۔

    متن:
    {text}

    درست شدہ متن:
    """
```

In [46]:
def correct_urdu_text(text):
    prompt = f"""
    آپ ایک ماہر اردو پروف ریڈر ہیں۔ آپ کا کام صرف ہجے کی غلطیاں اور علامات درست کرنا ہے۔

    ###ہدایات###
    1. صرف ہجے کی غلطیاں درست کریں۔
    2. صرف علامات، وقفے اور نقطہ ویرگول درست کریں۔
    3. تمام الفاظ، جملے، اور افعال بالکل ویسے کے ویسے رہیں۔  
    - جیسے "کر دیا" یا "بتائیں گے" کو کبھی نہ بدلیں۔
    4. صرف درست شدہ اردو متن واپس کریں۔  
    - کوئی وضاحت، تبصرہ یا اضافی متن شامل نہ کریں۔
    5. جہاں ممکن ہو، اصل فارمیٹنگ اور لائن بریکس برقرار رکھیں۔
    6. ایسے بے معنی یا ٹوٹے ہوئے الفاظ کو سیاق و سباق کے مطابق درست اور بامعنی لفظ میں تبدیل کرنا  

    آپ کو متن کی اصل ساخت، اسلوب اور معنی کو برقرار رکھتے ہوئے صرف ضروری درستی کرنی ہے۔

    ###تفصیلی سوچ (Chain of Thoughts)###
    درج ذیل سخت عمل کریں:

    1. متن کو لفظ بہ لفظ اسکین کریں۔
    2. ہر لفظ کو الگ الگ دیکھیں:
    - اگر ہجے کی غلطی ہو یا سیاق و سباق میں بے معنی ہوں تو صرف وہی لفظ درست کریں۔
    - اگر لفظ درست ہو تو اسے ہرگز تبدیل نہ کریں۔
    3. سیاق و سباق کو سمجھ کر ممکنہ درست لفظ کا انتخاب کریں
     جملے کی ساخت، زمانہ، فعل یا اسلوب کو نہ چھیڑیں۔
    4. صرف punctuation (۔ ، ؛ ؟) درست کریں۔
    5. کسی بھی جملے کو دوبارہ نہ لکھیں۔
    6. کسی بھی لفظ کو مترادف یا بہتر شکل میں تبدیل نہ کریں۔
    7. آخر میں اصل متن اور درست شدہ متن کا اندرونی طور پر موازنہ کریں:
    - اگر کسی درست لفظ کو تبدیل کیا گیا ہو تو اسے واپس اصل شکل میں کر دیں۔
    8. صرف حتمی درست شدہ متن واپس کریں۔

    ###کیا نہ کریں###
    - جملے کو دوبارہ نہ لکھیں۔
    - فعل کی شکل نہ بدلیں۔
    - زمانہ تبدیل نہ کریں۔
    - وضاحت شامل نہ کریں۔
    - heading شامل نہ کریں۔
    - "درست شدہ متن:" دوبارہ نہ لکھیں۔
    - کوئی اضافی لائن شامل نہ کریں۔

    متن:
    {text}

    درست شدہ متن:
    """

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    output = response[0]["generated_text"]

    # Extract corrected text
    if "درست شدہ متن:" in output:
        output = output.split("درست شدہ متن:")[-1].strip()

    return output

In [47]:
line = "*" * 20

print(f"{line}\nSpeech To Text\n{line}")

base_path = r"D:\Anum\ASR_Urdu\Audios"
output_file = r"D:\Anum\ASR_Urdu\transcriptions.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Urdu ASR Transcriptions\n")
    f.write("=" * 60 + "\n\n")

with open(output_file, "a", encoding="utf-8") as f:

    for audio in os.listdir(base_path):

        audio_input = os.path.join(base_path, audio)
        print(audio_input)

        if not os.path.exists(audio_input):
            print("Audio file not found.")
            continue

        print("Processing Audio File...")

        urdu_text = pipe_lora(
            audio_input,
            generate_kwargs={"language": "urdu"}
        )['text']

        corrected_text = correct_urdu_text(urdu_text)

        print(f"LoRA Fine-tuned Model Detected Speech : \n{urdu_text}")
        print(f"\nCorrected Text:\n{corrected_text}")
        print("-" * 50)

        # Write to file
        f.write(f"{'='*60}\n")
        f.write(f"FILE: {audio}\n")
        f.write(f"{'-'*60}\n")
        f.write(f"RAW TEXT:\n{urdu_text}\n\n")
        f.write(f"CORRECTED TEXT:\n{corrected_text}\n")
        f.write(f"{'='*60}\n\n")

********************
Speech To Text
********************
D:\Anum\ASR_Urdu\Audios\Big_News.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

    ###نوٹ###
    - یہ ایک بہت ہی سادہ اور آسان کام ہے۔
    - اسے ایک چیلنج بنانے کے لئے، ہم نے کچھ بے معنی اور ٹوٹے ہوئے الفاظ شامل کیے ہیں جو آپ کو سیاق و سباق کے مطابق درست کرنے کی ضرورت ہے۔
    - اسے ایک چیلنج بنانے کے ل
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Big_News.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔

    ###نوٹ###
    - یہ ایک بہت ہی سادہ اور آسان کام ہے۔
    - اسے ایک چیلنج بنانے کے لئے، ہم نے کچھ بے معنی اور ٹوٹے ہوئے الفاظ شامل کیے ہیں جو آپ کو سیاق و سباق کے مطابق درست کرنے کی ضرورت ہے۔
    - اسے ایک چیلنج بنانے کے ل
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Former_US_Secretary.mp3
Processing Audio File...


...

LoRA Fine-tuned Model Detected Speech : 
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے

Corrected Text:
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لئے تیل کی قیمتوں میں اضافہ کا اعلان کیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کیا ہے۔ پیٹرول فی لیٹر کی قیمت 3.05 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لئے تیل کی قیمتوں میں اضافہ کا اعلان کیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کیا ہے۔ پیٹرول فی لیٹر کی قیمت 3.05 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی میں 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------


# Prompt 03 
---
```python
prompt = f"""
    You are an expert Urdu proofreader and spelling corrector.

    Task:
    Correct the provided Urdu text by fixing ALL spelling mistakes, typographical errors, punctuation errors, and incorrect word forms.

    Instructions:
    - Carefully review every word and correct all spelling errors.
    - Fix incorrect words such as misspelled names, places, or common Urdu terms.
    - Correct punctuation and sentence boundaries where needed.
    - Keep the original meaning exactly the same.
    - Do not rewrite or paraphrase the text.
    - Do not add or remove information.
    - If a word is clearly misspelled, replace it with the correct standard Urdu spelling.
    - If the text is already correct, return it unchanged.

    Text:
    {text}

    Output:
    """
```

In [48]:
# Correction Function
def correct_urdu_text(text):
    prompt = f"""
    You are an expert Urdu proofreader and spelling corrector.

    Task:
    Correct the provided Urdu text by fixing ALL spelling mistakes, typographical errors, punctuation errors, and incorrect word forms.

    Instructions:
    - Carefully review every word and correct all spelling errors.
    - Fix incorrect words such as misspelled names, places, or common Urdu terms.
    - Correct punctuation and sentence boundaries where needed.
    - Keep the original meaning exactly the same.
    - Do not rewrite or paraphrase the text.
    - Do not add or remove information.
    - If a word is clearly misspelled, replace it with the correct standard Urdu spelling.
    - If the text is already correct, return it unchanged.

    Text:
    {text}

    Output:
    """

    response = chatbot(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    output = response[0]["generated_text"]

    
    # Extract corrected text
    if "Output:" in output:
        output = output.split("Output:")[-1].strip()

    return output

In [49]:
line = "*" * 20

print(f"{line}\nSpeech To Text\n{line}")

base_path = r"D:\Anum\ASR_Urdu\Audios"
output_file = r"D:\Anum\ASR_Urdu\transcriptions.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("Urdu ASR Transcriptions\n")
    f.write("=" * 60 + "\n\n")

with open(output_file, "a", encoding="utf-8") as f:

    for audio in os.listdir(base_path):

        audio_input = os.path.join(base_path, audio)
        print(audio_input)

        if not os.path.exists(audio_input):
            print("Audio file not found.")
            continue

        print("Processing Audio File...")

        urdu_text = pipe_lora(
            audio_input,
            generate_kwargs={"language": "urdu"}
        )['text']

        corrected_text = correct_urdu_text(urdu_text)

        print(f"LoRA Fine-tuned Model Detected Speech : \n{urdu_text}")
        print(f"\nCorrected Text:\n{corrected_text}")
        print("-" * 50)

        # Write to file
        f.write(f"{'='*60}\n")
        f.write(f"FILE: {audio}\n")
        f.write(f"{'-'*60}\n")
        f.write(f"RAW TEXT:\n{urdu_text}\n\n")
        f.write(f"CORRECTED TEXT:\n{corrected_text}\n")
        f.write(f"{'='*60}\n\n")

********************
Speech To Text
********************
D:\Anum\ASR_Urdu\Audios\Big_News.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے، پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے۔ پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Big_News.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی

Corrected Text:
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے، پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے۔ پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\Former_US_Secretary.mp3
Processing Audio File...


...

LoRA Fine-tuned Model Detected Speech : 
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے

Corrected Text:
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش گوئی کر دی گئی ہے۔ محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے۔ محکمہ موسمیات کے ادارے کے مطابق شہر حیدرآباد میں اگلے چار دنوں تک بارش ہو سکتی ہے۔ کرونا کے بھرپور اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگاتار اپیل کی جا رہی ہے۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.mp3
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے۔ پیٹرول فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے، جو پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی کے برابر 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------
D:\Anum\ASR_Urdu\Audios\UAE_oil_prices.wav
Processing Audio File...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA Fine-tuned Model Detected Speech : 
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے

Corrected Text:
یہاں آپ کو بتائیں گے کہ متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے۔ فیول پرائس کمیٹی نے پیٹرول کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے۔ پیٹرول فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے، جو پاکستانی کرنسی میں 216 روپے 87 پیسے کے برابر ہے۔ ڈیزل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی ہے، جو پاکستانی کرنسی کے برابر 240 روپے 33 پیسے کے برابر ہے۔
--------------------------------------------------
